In [ ]:
from bs4 import BeautifulSoup as bs 
import pandas as pd
import requests
import re
import nltk


In [ ]:

nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng') # For Part-of-Speech tagging
nltk.download('maxent_ne_chunker_tab')          # For Entity recognition
nltk.download('words')                          # Dictionary to help identify names

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Peter\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Peter\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     C:\Users\Peter\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\Peter\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [3]:
def join_text_elements(text_elements: list[str]) -> str:
    """
    Helper function to join text extracted from <p> tags and links.

    Will join the text with a whitespace, unless the proceeding text ends with a punctuation mark, in which case it will join without a whitespace.
    This is to handle cases like "Draftsim's" where the apostrophe is part of the word and should not be separated by a space. 

    :param text_elements: A list of strings extracted from the HTML content, including both regular text and link text.
    """
    joined_text = ""
    for i, element in enumerate(text_elements):
        next_element = text_elements[i + 1] if i < len(text_elements) - 1 else ""
        prev_element = text_elements[i - 1] if i > 0 else ""
        if (
            next_element.startswith("'")
            or next_element.startswith(",")
            or next_element.startswith(";")
            or next_element.startswith(".")
            or next_element.startswith(":")
            or next_element.startswith("!")
            or next_element.startswith("?")
            or next_element.startswith("‘")  # stylistic apostrophe because of course this is a thing
            or prev_element.endswith("-")  # handle cases like "non-Entity"
        ):
            joined_text += element
        else:
            joined_text += element   + " "
    
    return joined_text.strip()

In [4]:
def scrape_words_for_ner(url):
    # 1. Fetch the webpage
    # We use a User-Agent header because some sites block default Python requests
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to retrieve page. Status code: {response.status_code}")
        return []

    # 2. Parse the HTML using BeautifulSoup
    soup = bs(response.text, 'html.parser')

    # 3. Locate the main article content
    # Draftsim and most WordPress sites use 'entry-content' or the <article> tag.
    main_content = soup.find('article') 
    if not main_content:
        # Fallback if <article> isn't found
        main_content = soup.find('div', class_='entry-content')
        
    if not main_content:
        print("Could not find the main content container.")
        return []

    # 4. Extract text from paragraphs and headings
    # We ignore <a> tags in menus, footer text, etc., by staying inside main_content
    text_elements = main_content.find_all(['p', 'h1', 'h2', 'h3'])
    
    raw_text = []
    for element in text_elements:
        # Get the text and strip leading/trailing whitespace
        clean_text = element.get_text(strip=True)
        if clean_text:
            raw_text.append(clean_text)

    # Join the extracted paragraphs into one large string
    full_text = " ".join(raw_text)

    # 5. Clean up the text (Optional but recommended for NER)
    # Remove extra spaces or weird unicode characters if necessary
    full_text = re.sub(r'\s+', ' ', full_text)

    # 6. Tokenize the text into words
    # NLTK's word_tokenize is great for NER because it separates punctuation from words 
    # (e.g., "Relentless," becomes ["Relentless", ","])
    words = nltk.word_tokenize(full_text)

    return words



In [5]:
def join_text_elements(text_elements: list[str]) -> str:
    """
    Helper function to join text extracted from <p> tags and links.

    Will join the text with a whitespace, unless the proceeding text ends with a punctuation mark, in which case it will join without a whitespace.
    This is to handle cases like "Draftsim's" where the apostrophe is part of the word and should not be separated by a space. 

    :param text_elements: A list of strings extracted from the HTML content, including both regular text and link text.
    """
    joined_text = ""
    for i, element in enumerate(text_elements):
        next_element = text_elements[i + 1] if i < len(text_elements) - 1 else ""
        prev_element = text_elements[i - 1] if i > 0 else ""
        if (
            next_element.startswith("'")
            or next_element.startswith(",")
            or next_element.startswith(";")
            or next_element.startswith(".")
            or next_element.startswith(":")
            or next_element.startswith("!")
            or next_element.startswith("?")
            or next_element.startswith("‘")  # stylistic apostrophe because of course this is a thing
            or prev_element.endswith("-")  # handle cases like "non-Entity"
        ):
            joined_text += element
        else:
            joined_text += element   + " "
    
    return joined_text.strip()


def scrape_words_for_ner(url):
    # 1. Fetch the webpage
    # We use a User-Agent header because some sites block default Python requests
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to retrieve page. Status code: {response.status_code}")
        return []

    # 2. Parse the HTML using BeautifulSoup
    soup = bs(response.text, 'html.parser')

    # 3. Locate the main article content
    # Draftsim and most WordPress sites use 'entry-content' or the <article> tag.
    main_content = soup.find('article') 
    if not main_content:
        # Fallback if <article> isn't found
        main_content = soup.find('div', class_='entry-content')
        
    if not main_content:
        print("Could not find the main content container.")
        return []

    text_elements = main_content.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6'])
    links = [element.find_all(['a']) for element in text_elements]
    
    full_text = []
    for text_element in text_elements:
        text_element_text = []
        for child in text_element.contents:
            if child.name == 'a' or child.name in ['b', 'i', 'strong', 'em', 'span']:  # handle common inline elements that may contain text
                text_element_text.append(child.get_text(strip=True))
            elif isinstance(child, str):
                clean_text = child.strip()
                if clean_text:
                    text_element_text.append(clean_text)
        joined_text = join_text_elements(text_element_text)
        full_text.append(joined_text)

    full_text = "\n\n".join(full_text)
    full_text = full_text.replace('|', ' ')  # Remove pipe characters that are used in some articles to separate sections, as they can interfere with NER processing
    words = nltk.word_tokenize(full_text)
    return words

___Artifacts___

In [13]:
# Run the scraper
url = 'https://draftsim.com/best-artifacts-mtg/'
words_list2 = scrape_words_for_ner(url)

# Display the first 50 words to verify
print(f"Total words extracted: {len(words_list2)}\n")
print(words_list2)


Total words extracted: 7383

['Last', 'updated', 'on', 'November', '1', ',', '2025', 'Mana', 'Crypt', 'Illustration', 'by', 'Dominik', 'Mayer', 'Artifacts', 'are', 'as', 'integral', 'to', 'Magic', 'as', 'creatures', 'and', 'lands', ',', 'and', 'they', '’', 've', 'been', 'around', 'since', 'the', 'beginning', '.', 'They', '’', 're', 'essentials', 'in', 'every', 'new', 'Magic', 'set', ',', 'sometimes', 'with', 'entire', 'planes', 'or', 'blocks', 'use', 'artifacts', 'as', 'a', 'central', 'theme', '.', 'Mirrodin', 'block', ',', 'Kaladesh', 'block', ',', 'and', 'Antiquities', 'are', 'just', 'a', 'few', 'examples', 'of', 'sets', 'that', 'put', 'artifacts', 'front', 'and', 'center', '.', 'They', 'also', 'are', 'a', 'driving', 'force', 'in', 'the', 'lore', '.', 'Mirari', ',', 'The', 'Immortal', 'Sun', ',', 'Helvault', ',', 'Legacy', 'Weapon', ',', 'and', 'The', 'Aetherspark', '…', 'the', 'list', 'continues', '.', 'Feels', 'like', 'we', 'always', 'travel', 'to', 'new', 'planes', 'in', 'search',

In [14]:
import pandas as pd
df = pd.DataFrame({'Token':words_list2})
df.to_csv('Artifacts_df.csv')

___Creatures___

In [15]:
# Run the scraper
url = "https://draftsim.com/mtg-best-creatures/"
words_list3 = scrape_words_for_ner(url)

# Display the first 50 words to verify
print(f"Total words extracted: {len(words_list3)}\n")
print(words_list3[:50])

Total words extracted: 5160

['Last', 'updated', 'on', 'April', '13', ',', '2026', 'Colossal', 'Dreadmaw', 'Illustration', 'by', 'Jesper', 'Ejsing', 'Creatures', 'are', 'vital', 'to', 'playing', 'Magic', '.', 'Even', 'decks', 'focused', 'on', 'other', 'card', 'types', 'use', 'a', 'few', 'creatures', 'to', 'help', 'impact', 'the', 'board', '.', 'The', 'few', 'devoutly', 'creatureless', 'decks', ',', 'like', 'many', 'iterations', 'of', 'Pioneer', 'Azorius', 'Control']


In [16]:
df = pd.DataFrame({'Token':words_list3})
df.to_csv('Creatures_df.csv')

___Non-Basic Lands___

In [6]:
# Run the scraper
url = "https://magic.wizards.com/en/news/feature/top-50-nonbasic-lands-2003-04-04"
words_list4 = scrape_words_for_ner(url)

# Display the first 50 words to verify
print(f"Total words extracted: {len(words_list4)}\n")
print(words_list4[:50])

Total words extracted: 4982

['The', 'Top', '50', 'Nonbasic', 'Lands', ',', 'has', 'it', 'been', 'two', 'weeks', 'already', 'since', 'my', 'last', 'article', '?', 'Now', 'we', "'ve", 'gone', 'from', 'Card', 'Drawing', 'to', 'Nonbasic', 'Land', 'Week', 'here', 'at', 'MagicTheGathering.com', ',', 'and', 'let', 'me', 'tell', 'you', 'I', 'could', "n't", 'be', 'happier', 'about', 'it', '!', 'After', 'all', ',', 'one', 'of']


In [9]:
df = pd.DataFrame({'Token':words_list4})
df.to_csv('Lands_df.csv')

___Enchantment___

In [10]:
# Run the scraper
url = "https://draftsim.com/mtg-enchantment/"
words_list5 = scrape_words_for_ner(url)

# Display the first 50 words to verify
print(f"Total words extracted: {len(words_list5)}\n")
print(words_list5[:50])

Total words extracted: 4723

['Last', 'updated', 'on', 'September', '3', ',', '2025', 'Eldrazi', 'Conscription', 'Illustration', 'by', 'Jaime', 'Jones', 'Enchantments', 'have', 'been', 'in', 'MTG', 'since', 'the', 'beginning', ',', 'and', 'they', '’', 're', 'one', 'of', 'the', 'coolest', 'build-arounds', 'in', 'the', 'game', '.', 'They', '’', 're', 'hard', 'to', 'remove', 'and', 'enable', 'countless', 'combos', 'and', 'interactions', '.', 'Enchantments', 'usually']


In [11]:
df = pd.DataFrame({'Token':words_list5})
df.to_csv('Enchantment_df.csv')

___Instant___

In [13]:
# Run the scraper
url = "https://draftsim.com/mtg-instant/"
words_list6 = scrape_words_for_ner(url)

# Display the first 50 words to verify
print(f"Total words extracted: {len(words_list6)}\n")
print(words_list6[:50])

Total words extracted: 6413

['Last', 'updated', 'on', 'April', '2', ',', '2026', 'Force', 'of', 'Will', 'Illustration', 'by', 'Donato', 'Giancola', 'Instant', 'spells', 'in', 'MTG', 'are', 'some', 'of', 'the', 'most', 'exciting', 'cards', 'to', 'play', 'simply', 'because', 'they', 'let', 'you', 'and', 'your', 'opponents', 'surprise', 'and', 'outwit', 'each', 'other', '.', 'Instants', 'turn', 'combat', 'around', 'and', 'make', 'players', 'feel', 'uneasy']


In [14]:
df = pd.DataFrame({'Token':words_list6})
df.to_csv('Insant_df.csv')

___Sorcery___

In [7]:
url = 'https://draftsim.com/mtg-sorcery/'
words_list7 = scrape_words_for_ner(url)

# Display the first 50 words to verify
print(f"Total words extracted: {len(words_list7)}\n")
print(words_list7[:50])

Total words extracted: 6903

['Last', 'updated', 'on', 'April', '8', ',', '2026', 'Toxic', 'Deluge', 'Illustration', 'by', 'Svetlin', 'Velinov', 'A', 'ranking', 'of', 'a', 'fundamental', 'part', 'of', 'Magic', '?', 'What', 'kind', 'of', 'sorcery', 'is', 'this', '?', 'Sorceries', 'have', 'been', 'in', 'the', 'game', 'since', 'Alpha', 'and', 'are', 'foundational', 'to', 'Magic', '.', 'The', 'entire', 'game', 'revolves', 'around', 'sorcery-', 'versus']


In [8]:
df = pd.DataFrame({'Token':words_list7})
df.to_csv('Sorcery_df.csv')

___Planeswalker___

In [17]:
url = 'https://draftsim.com/best-planeswalkers-mtg/'
words_list8 = scrape_words_for_ner(url)

# Display the first 50 words to verify
print(f"Total words extracted: {len(words_list8)}\n")
print(words_list8[:50])

Total words extracted: 5424

['Last', 'updated', 'on', 'April', '16', ',', '2026', 'Ugin', ',', 'the', 'Spirit', 'Dragon', 'Illustration', 'by', 'Raymond', 'Swanland', 'Magic', 'was', 'founded', 'on', 'the', 'premise', 'that', 'you', 'and', 'your', 'opponent', 'are', 'powerful', 'dueling', 'planeswalkers', ',', 'engaged', 'in', 'glorious', 'magical', 'combat', ',', 'slinging', 'spells', 'and', 'summoning', 'creatures', 'to', 'aid', 'in', 'your', 'struggle', '.', 'The']


In [ ]:
df = pd.DataFrame({'Token':words_list8})
df.to_csv('Planeswalker_df.csv')

___Basic Article___

In [19]:
url = 'https://draftsim.com/mtg-secrets-of-strixhaven-combos/'
words_list9 = scrape_words_for_ner(url)

# Display the first 50 words to verify
print(f"Total words extracted: {len(words_list9)}\n")
print(words_list9[:50])

Total words extracted: 1751

['Last', 'updated', 'on', 'April', '20', ',', '2026', 'Planar', 'Engineering', 'Illustration', 'by', 'Liiga', 'Smilshkalne', 'Much', 'like', 'Avatar', ':', 'The', 'Last', 'Airbender', ',', 'Secrets', 'of', 'Strixhaven', 'looks', 'like', 'the', 'type', 'of', 'set', 'design', 'that', 'lends', 'itself', 'to', 'combos', 'quite', 'easily', '.', 'There', '’', 's', 'a', 'lot', 'of', 'blink', ',', 'mana', 'generation', ',']


In [20]:
df = pd.DataFrame({'Token':words_list9})
df.to_csv('BasicArticle_df.csv')